# 🤖 AI Data Analyst Chatbot
### Powered by Groq API (llama-3.3-70b-versatile)

---

## 📊 What This Notebook Does:

This is a complete AI-powered data analysis assistant that:

✅ **Analyzes Datasets**: Automatically examines shape, columns, data types, missing values

✅ **Provides Insights**: Identifies patterns, correlations, anomalies, and trends

✅ **Generates Visualizations**: Creates and recommends charts (matplotlib, seaborn, plotly)

✅ **Answers Questions**: Natural language interface for data exploration

✅ **Professional Reports**: Structured responses with Summary → Insights → Code → Recommendations

---

### 🎯 How to Use:
1. Run all cells in order
2. Load your dataset (CSV, Excel, or JSON)
3. Ask questions in natural language
4. Get instant insights and visualization code

---

## 📦 Step 1: Install Required Packages

Run this cell once to install all dependencies:

In [ ]:
# Uncomment and run if packages are not installed
# !pip install groq python-dotenv pandas numpy matplotlib seaborn plotly openpyxl -q

## 📚 Step 2: Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from groq import Groq
from dotenv import load_dotenv
import json
import warnings
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')

# Set visualization styles
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ All packages imported successfully!")
print("📊 Visualization settings configured")

## 🤖 Step 3: Define AI Data Analyst Chatbot Class

This cell contains the complete chatbot implementation:

In [ ]:
class DataAnalystChatbot:
    """AI-powered Data Analyst using Groq API"""
    
    def __init__(self, api_key=None):
        # Load API key from environment or parameter
        load_dotenv()
        self.api_key = api_key or os.getenv("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("GROQ_API_KEY not found. Please set it in .env file or pass as parameter.")
        
        self.client = Groq(api_key=self.api_key)
        self.model = "llama-3.3-70b-versatile"
        self.conversation_history = []
        self.current_dataset = None
        self.dataset_info = {}
        
    def load_dataset(self, file_path):
        """Load dataset from CSV, Excel, or JSON"""
        try:
            if file_path.endswith('.csv'):
                self.current_dataset = pd.read_csv(file_path)
            elif file_path.endswith(('.xlsx', '.xls')):
                self.current_dataset = pd.read_excel(file_path)
            elif file_path.endswith('.json'):
                self.current_dataset = pd.read_json(file_path)
            else:
                return "❌ Unsupported file format. Please use CSV, Excel, or JSON."
            
            self._analyze_dataset()
            return f"✅ Dataset loaded successfully!\n📊 Shape: {self.current_dataset.shape[0]} rows × {self.current_dataset.shape[1]} columns"
        except Exception as e:
            return f"❌ Error loading dataset: {str(e)}"
    
    def _analyze_dataset(self):
        """Perform comprehensive dataset analysis"""
        df = self.current_dataset
        
        # Basic info
        self.dataset_info = {
            "shape": df.shape,
            "columns": df.columns.tolist(),
            "dtypes": {str(k): str(v) for k, v in df.dtypes.to_dict().items()},
            "missing_values": df.isnull().sum().to_dict(),
            "missing_percentage": (df.isnull().sum() / len(df) * 100).to_dict(),
            "numeric_columns": df.select_dtypes(include=[np.number]).columns.tolist(),
            "categorical_columns": df.select_dtypes(include=['object', 'category']).columns.tolist(),
            "memory_usage": df.memory_usage(deep=True).sum() / 1024**2,  # MB
        }
        
        # Descriptive statistics
        if len(self.dataset_info["numeric_columns"]) > 0:
            desc_stats = df[self.dataset_info["numeric_columns"]].describe()
            self.dataset_info["descriptive_stats"] = desc_stats.to_dict()
        
        # Correlations
        if len(self.dataset_info["numeric_columns"]) > 1:
            corr_matrix = df[self.dataset_info["numeric_columns"]].corr()
            self.dataset_info["correlations"] = corr_matrix.to_dict()
            
            # Find strongest correlations
            corr_pairs = []
            for i in range(len(corr_matrix.columns)):
                for j in range(i+1, len(corr_matrix.columns)):
                    col1, col2 = corr_matrix.columns[i], corr_matrix.columns[j]
                    corr_val = corr_matrix.iloc[i, j]
                    if abs(corr_val) > 0.5:  # Strong correlation threshold
                        corr_pairs.append((col1, col2, corr_val))
            self.dataset_info["strong_correlations"] = corr_pairs
        
        # Detect potential outliers using IQR method
        outliers = {}
        for col in self.dataset_info["numeric_columns"]:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outlier_count = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
            if outlier_count > 0:
                outliers[col] = int(outlier_count)
        self.dataset_info["outliers"] = outliers
    
    def get_dataset_summary(self):
        """Generate comprehensive dataset summary"""
        if self.current_dataset is None:
            return "❌ No dataset loaded. Please load a dataset first."
        
        info = self.dataset_info
        
        summary = f"""
{'='*80}
📊 DATASET SUMMARY
{'='*80}

📐 SHAPE & STRUCTURE
   • Rows: {info['shape'][0]:,}
   • Columns: {info['shape'][1]}
   • Memory Usage: {info['memory_usage']:.2f} MB

📋 COLUMNS & DATA TYPES
"""
        for col, dtype in info['dtypes'].items():
            summary += f"   • {col}: {dtype}\n"
        
        summary += f"\n🔢 COLUMN CATEGORIES\n"
        summary += f"   • Numeric: {len(info['numeric_columns'])} columns\n"
        summary += f"   • Categorical: {len(info['categorical_columns'])} columns\n"
        
        # Missing values
        total_missing = sum(info['missing_values'].values())
        summary += f"\n❓ MISSING VALUES (Total: {total_missing})\n"
        if total_missing > 0:
            for col, missing in info['missing_values'].items():
                if missing > 0:
                    pct = info['missing_percentage'][col]
                    summary += f"   • {col}: {missing} ({pct:.2f}%)\n"
        else:
            summary += "   ✅ No missing values detected\n"
        
        # Outliers
        if info.get('outliers'):
            summary += f"\n⚠️ POTENTIAL OUTLIERS (IQR Method)\n"
            for col, count in info['outliers'].items():
                summary += f"   • {col}: {count} outliers\n"
        
        # Strong correlations
        if info.get('strong_correlations'):
            summary += f"\n🔗 STRONG CORRELATIONS (|r| > 0.5)\n"
            for col1, col2, corr in info['strong_correlations'][:5]:  # Top 5
                summary += f"   • {col1} ↔ {col2}: {corr:.3f}\n"
        
        summary += f"\n{'='*80}\n"
        return summary
    
    def chat(self, user_message):
        """Main chat interface with Groq API"""
        if self.current_dataset is None:
            return "❌ Please load a dataset first using load_dataset() method."
        
        # Build context
        context = self._build_context()
        
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        # Prepare messages for Groq API
        system_prompt = f"""You are an expert AI Data Analyst and Visualization Expert.

Your responsibilities:
- Analyze datasets thoroughly and provide actionable insights
- Identify patterns, trends, correlations, and anomalies
- Suggest appropriate visualizations for the data
- Generate clean, working Python code (matplotlib, seaborn, plotly)
- Explain results in simple, clear, professional English
- Provide data cleaning and preprocessing recommendations

Current Dataset Context:
{context}

ALWAYS structure your answers as:
1. 📊 Summary: Brief overview of findings
2. 💡 Insights: Key patterns and discoveries (3-5 bullet points)
3. 📈 Visualizations: Recommended charts (if applicable)
4. 💻 Code: Python code for analysis/visualization (if applicable)
5. 🎯 Recommendations: Actionable next steps

Be precise, professional, and actionable."""
        
        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(self.conversation_history[-10:])  # Last 10 messages
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.7,
                max_tokens=2500
            )
            
            assistant_message = response.choices[0].message.content
            
            self.conversation_history.append({
                "role": "assistant",
                "content": assistant_message
            })
            
            return assistant_message
            
        except Exception as e:
            return f"❌ Error communicating with Groq API: {str(e)}"
    
    def _build_context(self):
        """Build context string with dataset information"""
        if not self.dataset_info:
            return "No dataset loaded."
        
        info = self.dataset_info
        context = f"""
Dataset Shape: {info['shape'][0]} rows × {info['shape'][1]} columns
Columns: {', '.join(info['columns'])}
Numeric Columns ({len(info['numeric_columns'])}): {', '.join(info['numeric_columns'])}
Categorical Columns ({len(info['categorical_columns'])}): {', '.join(info['categorical_columns'])}
Total Missing Values: {sum(info['missing_values'].values())}
"""
        if info.get('strong_correlations'):
            context += f"Strong Correlations Found: {len(info['strong_correlations'])}\n"
        if info.get('outliers'):
            context += f"Columns with Outliers: {', '.join(info['outliers'].keys())}\n"
        
        return context
    
    def reset_conversation(self):
        """Reset conversation history"""
        self.conversation_history = []
        return "✅ Conversation history cleared."

print("✅ DataAnalystChatbot class defined successfully!")

## 🚀 Step 4: Initialize the Chatbot

In [ ]:
# Initialize the AI Data Analyst Chatbot
try:
    chatbot = DataAnalystChatbot()
    print("✅ AI Data Analyst Chatbot initialized successfully!")
    print(f"🤖 Model: {chatbot.model}")
    print(f"🔑 API Key: {'*' * 20}{chatbot.api_key[-10:]}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 Make sure your .env file contains: GROQ_API_KEY='your_key_here'")

## 📂 Step 5: Load Your Dataset

**Supported formats**: CSV, Excel (.xlsx, .xls), JSON

Replace `'your_dataset.csv'` with your actual file path:

In [ ]:
# 📁 Load your dataset
file_path = "your_dataset.csv"  # 👈 Change this to your file path

result = chatbot.load_dataset(file_path)
print(result)

# Display first few rows
if chatbot.current_dataset is not None:
    print("\n📋 First 5 rows:")
    display(chatbot.current_dataset.head())

### 🧪 Option: Create Sample Dataset for Testing

In [ ]:
# Uncomment to create and load a sample dataset
# np.random.seed(42)
# sample_df = pd.DataFrame({
#     'age': np.random.randint(18, 80, 200),
#     'income': np.random.randint(20000, 150000, 200),
#     'score': np.random.uniform(0, 100, 200),
#     'satisfaction': np.random.randint(1, 6, 200),
#     'category': np.random.choice(['A', 'B', 'C', 'D'], 200),
#     'region': np.random.choice(['North', 'South', 'East', 'West'], 200)
# })
# # Add some missing values
# sample_df.loc[np.random.choice(200, 15, replace=False), 'income'] = np.nan
# sample_df.to_csv('sample_data.csv', index=False)
# print("✅ Sample dataset created: sample_data.csv")
# 
# # Load it
# result = chatbot.load_dataset('sample_data.csv')
# print(result)
# display(chatbot.current_dataset.head())

## 📊 Step 6: Get Comprehensive Dataset Summary

In [ ]:
# Get detailed dataset summary
summary = chatbot.get_dataset_summary()
print(summary)

## 🔍 Step 7: Explore Dataset Details

In [ ]:
# Display dataset info
if chatbot.current_dataset is not None:
    df = chatbot.current_dataset
    
    print("📋 Dataset Info:")
    print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print("\n📊 Statistical Summary:")
    display(df.describe())
    
    print("\n🔢 Data Types:")
    display(df.dtypes.to_frame('dtype'))
    
    print("\n❓ Missing Values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Percentage': missing_pct
    })
    display(missing_df[missing_df['Missing Count'] > 0])
else:
    print("❌ No dataset loaded yet.")

## 💬 Step 8: Chat with AI Analyst

Now you can ask any questions about your data in natural language!

In [ ]:
# 💬 Ask your first question
question = "What are the key insights from this dataset?"

print(f"❓ Question: {question}\n")
print("="*80)
response = chatbot.chat(question)
print(response)
print("="*80)

## 🎯 Step 9: Example Analysis Questions

Try these common data analysis questions:

In [ ]:
# 🔗 Question 1: Correlations
print("❓ What are the strongest correlations in this dataset?\n")
print("="*80)
response = chatbot.chat("What are the strongest correlations in this dataset? Explain what they mean.")
print(response)
print("="*80)

In [ ]:
# 📈 Question 2: Visualization Recommendations
print("❓ What visualizations would you recommend?\n")
print("="*80)
response = chatbot.chat("What visualizations would you recommend for this data? Provide specific chart types and reasons.")
print(response)
print("="*80)

In [ ]:
# ❓ Question 3: Missing Data Strategy
print("❓ How should I handle missing values?\n")
print("="*80)
response = chatbot.chat("How should I handle the missing values? Suggest specific strategies for each column.")
print(response)
print("="*80)

In [ ]:
# ⚠️ Question 4: Anomaly Detection
print("❓ Are there any anomalies or outliers?\n")
print("="*80)
response = chatbot.chat("Are there any anomalies or outliers in the data? How should I handle them?")
print(response)
print("="*80)

In [ ]:
# 🎨 Question 5: Visualization Code
print("❓ Generate visualization code\n")
print("="*80)
response = chatbot.chat("Generate Python code to create a correlation heatmap and distribution plots for numeric columns.")
print(response)
print("="*80)

## 📊 Step 10: Generate Automatic Visualizations

Create comprehensive visualizations of your dataset:

In [ ]:
# 🔥 Correlation Heatmap
if chatbot.current_dataset is not None:
    df = chatbot.current_dataset
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 1:
        print("📊 Correlation Heatmap\n")
        plt.figure(figsize=(14, 10))
        corr_matrix = df[numeric_cols].corr()
        
        # Create mask for upper triangle
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdYlGn', 
                    center=0, fmt='.2f', square=True, linewidths=1,
                    cbar_kws={"shrink": 0.8})
        plt.title('Correlation Heatmap', fontsize=18, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ Not enough numeric columns for correlation analysis.")
else:
    print("❌ No dataset loaded.")

In [ ]:
# 📈 Distribution Plots for All Numeric Columns
if chatbot.current_dataset is not None:
    df = chatbot.current_dataset
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        print(f"📊 Distribution Plots for {len(numeric_cols)} Numeric Columns\n")
        
        n_cols = min(3, len(numeric_cols))
        n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5*n_rows))
        axes = axes.flatten() if len(numeric_cols) > 1 else [axes]
        
        for idx, col in enumerate(numeric_cols):
            if idx < len(axes):
                # Histogram with KDE
                axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', 
                              alpha=0.7, color='skyblue', density=True)
                
                # Add KDE line
                df[col].dropna().plot(kind='kde', ax=axes[idx], secondary_y=False, 
                                     color='red', linewidth=2)
                
                axes[idx].set_title(f'Distribution: {col}', fontsize=12, fontweight='bold')
                axes[idx].set_xlabel(col, fontsize=10)
                axes[idx].set_ylabel('Density', fontsize=10)
                axes[idx].grid(alpha=0.3)
        
        # Hide empty subplots
        for idx in range(len(numeric_cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ No numeric columns found.")
else:
    print("❌ No dataset loaded.")

In [ ]:
# 📦 Box Plots for Outlier Detection
if chatbot.current_dataset is not None:
    df = chatbot.current_dataset
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        print(f"📦 Box Plots for Outlier Detection\n")
        
        n_cols = min(3, len(numeric_cols))
        n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5*n_rows))
        axes = axes.flatten() if len(numeric_cols) > 1 else [axes]
        
        for idx, col in enumerate(numeric_cols):
            if idx < len(axes):
                sns.boxplot(y=df[col].dropna(), ax=axes[idx], color='lightcoral')
                axes[idx].set_title(f'Box Plot: {col}', fontsize=12, fontweight='bold')
                axes[idx].set_ylabel(col, fontsize=10)
                axes[idx].grid(alpha=0.3)
        
        # Hide empty subplots
        for idx in range(len(numeric_cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ No numeric columns found.")
else:
    print("❌ No dataset loaded.")

In [ ]:
# 📊 Missing Values Visualization
if chatbot.current_dataset is not None:
    df = chatbot.current_dataset
    missing = df.isnull().sum()
    
    if missing.sum() > 0:
        print("❓ Missing Values Visualization\n")
        
        missing_df = missing[missing > 0].sort_values(ascending=False)
        missing_pct = (missing_df / len(df) * 100).round(2)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # Bar plot - count
        missing_df.plot(kind='barh', ax=ax1, color='coral')
        ax1.set_title('Missing Values Count', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Count', fontsize=12)
        ax1.grid(alpha=0.3)
        
        # Bar plot - percentage
        missing_pct.plot(kind='barh', ax=ax2, color='skyblue')
        ax2.set_title('Missing Values Percentage', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Percentage (%)', fontsize=12)
        ax2.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print("✅ No missing values in the dataset!")
else:
    print("❌ No dataset loaded.")

## 💬 Step 11: Interactive Chat Session

Ask your own custom questions!

In [ ]:
# 💬 Ask a custom question
your_question = "What patterns do you see in this data?"  # 👈 Change this

print(f"❓ Your Question: {your_question}\n")
print("="*80)
response = chatbot.chat(your_question)
print(response)
print("="*80)

## 🎯 Step 12: Advanced Analysis Questions

Ask sophisticated data science questions:

In [ ]:
# 🎯 Advanced analysis questions
advanced_questions = [
    "What are the top 5 most important features and why?",
    "Suggest a complete data cleaning pipeline with code",
    "What machine learning models would work best for this data?",
    "Identify any data quality issues and how to fix them",
    "What feature engineering steps would you recommend?"
]

for q in advanced_questions:
    print(f"\n{'='*80}")
    print(f"❓ Question: {q}")
    print('='*80)
    response = chatbot.chat(q)
    print(response)
    print()

## 🔄 Step 13: Reset Conversation

Clear conversation history to start fresh:

In [ ]:
# Reset conversation history
result = chatbot.reset_conversation()
print(result)
print(f"\n📊 Dataset still loaded: {chatbot.current_dataset is not None}")
print(f"💬 Conversation messages: {len(chatbot.conversation_history)}")

## 🎨 Step 14: Custom Visualization Examples

Create specific visualizations based on your needs:

In [ ]:
# Ask AI to generate custom visualization code
viz_request = "Create a scatter plot matrix (pairplot) for all numeric columns with different colors for each category"

print(f"❓ Request: {viz_request}\n")
print("="*80)
response = chatbot.chat(viz_request)
print(response)
print("="*80)

In [ ]:
# Example: Pairplot (if you have categorical column)
if chatbot.current_dataset is not None:
    df = chatbot.current_dataset
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(numeric_cols) >= 2:
        print("📊 Pairplot of Numeric Variables\n")
        
        # Select first few numeric columns to avoid overcrowding
        cols_to_plot = numeric_cols[:4] if len(numeric_cols) > 4 else numeric_cols
        
        if len(categorical_cols) > 0:
            # Use first categorical column for hue
            hue_col = categorical_cols[0]
            sns.pairplot(df[list(cols_to_plot) + [hue_col]], hue=hue_col, 
                        diag_kind='kde', plot_kws={'alpha': 0.6})
        else:
            sns.pairplot(df[cols_to_plot], diag_kind='kde', plot_kws={'alpha': 0.6})
        
        plt.suptitle('Pairplot of Numeric Variables', y=1.02, fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ Need at least 2 numeric columns for pairplot.")
else:
    print("❌ No dataset loaded.")

## 📝 Step 15: Generate Full Analysis Report

Get a comprehensive analysis report:

In [ ]:
# Generate comprehensive analysis report
report_request = """
Generate a comprehensive data analysis report including:
1. Executive summary of key findings
2. Data quality assessment
3. Statistical insights and patterns
4. Correlation analysis
5. Outlier detection results
6. Recommendations for next steps
7. Suggested visualizations with code
"""

print("📊 GENERATING COMPREHENSIVE ANALYSIS REPORT\n")
print("="*80)
response = chatbot.chat(report_request)
print(response)
print("="*80)

## 🎓 Step 16: Example Use Cases

Common data analysis scenarios:

In [ ]:
# Use Case Examples
use_cases = {
    "Business Analytics": "What business insights can we derive from this data? Identify key metrics and trends.",
    "Predictive Modeling": "What would be the best approach to build a predictive model with this data?",
    "Customer Segmentation": "How can we segment customers based on this data? Suggest clustering approaches.",
    "Time Series": "If this data has temporal components, what time series analysis would you recommend?",
    "A/B Testing": "How would you set up an A/B test analysis framework for this data?"
}

# Pick one use case to explore
selected_use_case = "Business Analytics"  # 👈 Change this

if selected_use_case in use_cases:
    print(f"🎯 Use Case: {selected_use_case}\n")
    print("="*80)
    response = chatbot.chat(use_cases[selected_use_case])
    print(response)
    print("="*80)
else:
    print("Available use cases:")
    for uc in use_cases.keys():
        print(f"  - {uc}")

## 🚀 Step 17: Export Results

Save your analysis results:

In [ ]:
# Export conversation history
if chatbot.conversation_history:
    print("💾 Exporting conversation history...\n")
    
    # Save to text file
    with open('analysis_conversation.txt', 'w', encoding='utf-8') as f:
        f.write("AI DATA ANALYST CONVERSATION LOG\n")
        f.write("="*80 + "\n\n")
        
        for i, msg in enumerate(chatbot.conversation_history, 1):
            role = "YOU" if msg['role'] == 'user' else "AI ANALYST"
            f.write(f"[{i}] {role}:\n")
            f.write(msg['content'] + "\n")
            f.write("-"*80 + "\n\n")
    
    print("✅ Conversation saved to: analysis_conversation.txt")
    print(f"📊 Total messages: {len(chatbot.conversation_history)}")
else:
    print("⚠️ No conversation history to export.")

---

## 🎉 Congratulations!

You've completed the AI Data Analyst Chatbot tutorial!

### 📚 What You Can Do Next:

1. **Load your own dataset** and explore it
2. **Ask custom questions** specific to your analysis needs
3. **Generate visualizations** based on AI recommendations
4. **Export insights** for reports and presentations
5. **Iterate and refine** your analysis

### 💡 Tips:

- Be specific in your questions for better insights
- Ask for code examples when you need implementation help
- Use the conversation history to build on previous insights
- Reset conversation when switching to a new analysis focus

### 🔗 Resources:

- [Groq API Documentation](https://console.groq.com/docs)
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Seaborn Gallery](https://seaborn.pydata.org/examples/index.html)
- [Matplotlib Tutorials](https://matplotlib.org/stable/tutorials/index.html)

---

**Happy Analyzing! 📊🚀**